In [ ]:
from __future__ import annotations

import argparse
import json
import os
import tempfile
from dataclasses import asdict, dataclass, field
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "matplotlib"))

import joblib
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pywt
from skimage.color import rgb2gray
from skimage import exposure
from skimage.feature import graycomatrix, graycoprops
from skimage.io import imread
from skimage.transform import resize
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

from steelblast_classic_features import CLASS_NAMES, load_split

# SteelBlastQC Dataset
The SteelBlastQC dataset contains steel surfaces that are either shot-blasted or still need shot-blasting to achieve the required texture. The dataset includes “good” images (ready for paint) images and 766 “not-good” images (needs shot-blasting) images. As declared by the collaborating manufacturer, the ideally treated surface is clean and uniformly coarse with an average roughness level of SA 2.5. The “not-good” class presents several types of defects to the surface, located by industrial shot-blasting experts. These include: fresh welding lines and cuts, abrasion, corrosion, and discoloration.

# Load Dataset
This step reads training and test images from the structured dataset folders, collects file paths, and generates numeric labels for each image class.
It does not load image arrays immediately; instead it creates a lightweight image path list that is later used for preprocessing, normalization, feature extraction, and classification.


In [ ]:
@dataclass(frozen=True)
class AppConfig:
    dataset_dir: Path = Path("doi-10.34894-ekznn0(1)/SteelBlastQC")
    output_model: Path = Path("steelblast_svm_glcm_dwt.joblib")
    metrics_json: Path = Path("steelblast_svm_glcm_dwt_metrics.json")

app_config = AppConfig()

train_paths, y_train = load_split(app_config.dataset_dir, "train")
test_paths, y_test = load_split(app_config.dataset_dir, "test")




print(f"Loaded {len(train_paths)} training images and {len(test_paths)} test images.")

# `GLCM` + `DWT` feature pipeline
Handcrafted texture features are data-efficient and interpretable for small/industrial image sets; combining co-occurrence (GLCM) and multi-scale decomposition (DWT) captures complementary texture information while avoiding deep-learning data/compute demands.
- **Input & Resize**: `image_size = 256` — preserves texture detail while bounding computational cost and standardizing feature dimensions for GLCM/DWT.

- **Color & Illumination**: grayscale conversion with `illumination_normalization = "clahe"` (`clahe_clip_limit = 0.01`) and percentile fallback `(2.0, 98.0)`. CLAHE reduces lighting variation yet preserves local texture; percentile clipping provides robust global contrast control.

- **GLCM Design**:
    - `glcm_levels = 32` — limits matrix size and noise sensitivity while retaining characteristic texture gradients.
    - `glcm_distances = (1,2,4,8)` and `glcm_angles = (0, π/4, π/2, 3π/4)` — multi-scale, multi-orientation sampling captures varied blast textures.
    - `glcm_properties = ("contrast","dissimilarity","homogeneity","energy","correlation","ASM")` — summarize local variation, uniformity, and correlation.

- **DWT Design**:
    - `dwt_wavelet = "db2"`, `dwt_level = 3` — Daubechies wavelets provide compact time-frequency localization; three levels capture coarse-to-fine texture without exploding feature count.
    - `describe_coefficients(...)` uses mean, std, absolute moments, energy, entropy, and percentiles to robustly summarize coefficient distributions.

- **Feature Composition & Preprocessing**: concatenate GLCM + DWT features; `StandardScaler` standardizes features because SVMs depend on distances. `quantize_image(...)` stabilizes GLCM inputs.


In [ ]:
from steelblast_classic_features import (
    FeatureExtractionConfig,
    extract_features_from_image,
    load_split,
    normalize_illumination,
    read_grayscale_image,
)

# Loops through all images
# Extracts features
# Stacks them into matrix:
def build_feature_matrix(
    image_paths: list[Path],
    config: FeatureExtractionConfig,
    split_name: str,
) -> np.ndarray:
    features = []

    for index, image_path in enumerate(image_paths, start=1):
        features.append(extract_features(image_path, config))
        if index % 100 == 0 or index == len(image_paths):
            print(f"{split_name}: extracted {index}/{len(image_paths)} images")

    return np.vstack(features)

def balanced_limit(
    image_paths: list[Path],
    labels: np.ndarray,
    limit_per_class: int,
) -> tuple[list[Path], np.ndarray]:
    selected_paths: list[Path] = []
    selected_labels: list[int] = []

    for label in sorted(np.unique(labels)):
        class_indices = np.flatnonzero(labels == label)[:limit_per_class]
        selected_paths.extend(image_paths[index] for index in class_indices)
        selected_labels.extend(labels[index] for index in class_indices)

    return selected_paths, np.asarray(selected_labels, dtype=np.int64)

def extract_features(image_path: Path, config: FeatureExtractionConfig) -> np.ndarray:
    image = read_grayscale_image(image_path, config.image_size, config)
    return extract_features_from_image(image, config)

feature_config = FeatureExtractionConfig(
    image_size=256, # image size to resize to (256x256)
    illumination_normalization="clahe", # illumination normalization method
    glcm_levels=32, # Number of gray levels for GLCM quantization
    glcm_properties=("contrast",
                      "dissimilarity",
                        "homogeneity",
                          "energy",
                            "correlation", "ASM"), # GLCM properties to compute
    dwt_wavelet="db2", # Wavelet type for DWT
    dwt_level=3 # Decomposition level for DWTs
)


print(f"Train images: {len(train_paths)}")
print(f"Test images:  {len(test_paths)}")
print(f"Feature extraction config: {feature_config}")

#TODO # Split train into train and validation
X_train = build_feature_matrix(train_paths, feature_config, "train")
X_test = build_feature_matrix(test_paths, feature_config, "test")
 

# Modelling and validation
- **Model Choice & Pipeline**: `Pipeline([("scaler", StandardScaler()), ("svm", CalibratedClassifierCV(SVC(kernel="rbf", class_weight="balanced"), ensemble=False))])` — the RBF SVM handles non-linear boundaries on moderate feature vectors; `class_weight="balanced"` mitigates class imbalance; calibration provides probability outputs without relying on the deprecated `SVC(probability=True)` path.

- **Hyperparameter Ranges**:
    - `svm__estimator__C = [0.1, 1, 10, 100]` — explores under/over-regularization regimes.
    - `svm__estimator__gamma = ["scale", 0.01, 0.001, 0.0001]` — includes heuristic and small numeric bandwidths; coarse grid reduces search cost for moderate datasets.

- **Cross-Validation & Training Strategy**:
    - `StratifiedKFold(n_splits = min(max_cv_folds, min_class_count), shuffle=True, random_state=42)` — preserves class proportions and avoids invalid fold counts for the outer model-selection loop.
    - `CalibratedClassifierCV(..., method="sigmoid", ensemble=False, cv=3)` — learns probability calibration on top of the SVM margins so downstream confidence scores remain comparable to the transfer-learning path.
    - `scoring="f1"` and `class_weight="balanced"` prioritize balanced precision/recall for defect detection.




In [ ]:
@dataclass(frozen=True)
class TrainingConfig:
    param_grid: dict[str, list[object]] = field(
        default_factory=lambda: {
            "svm__estimator__C": [0.1, 1, 10, 100],
            "svm__estimator__gamma": ["scale", 0.01, 0.001, 0.0001],
        }
    )
    max_cv_folds: int = 5
    scoring: str = "f1"
    random_state: int = 42
    calibrated_cv_folds: int = 3
    # Parallel jobs for GridSearchCV. Use -1 outside restricted environments.
    n_jobs: int = 1
    verbose: int = 2


training_config = TrainingConfig()

# train model using SVM (RBF kernel)
def train_model(X_train: np.ndarray, y_train: np.ndarray, config: TrainingConfig) -> GridSearchCV:
    min_class_count = int(np.bincount(y_train).min())
    outer_splits = min(config.max_cv_folds, min_class_count)
    if outer_splits < 2:
        raise ValueError("Each class needs at least two training images.")

    calibration_splits = min(config.calibrated_cv_folds, min_class_count)
    if calibration_splits < 2:
        raise ValueError("Each class needs at least two training images for probability calibration.")

    base_svm = SVC(kernel="rbf", class_weight="balanced")
    calibrated_svm = CalibratedClassifierCV(
        estimator=base_svm,
        method="sigmoid",
        cv=calibration_splits,
        ensemble=False,
    )
    pipeline = Pipeline(
        steps=[
            ("scaler", StandardScaler()), # StandardScaler standardizes features by removing the mean and scaling to unit variance, which is important for SVM performance since it relies on distances in feature space. This ensures that all features contribute equally to the model and prevents features with larger scales from dominating the decision boundary.
            ("svm", calibrated_svm), # Use an explicitly calibrated SVM so predict_proba remains available after sklearn removes SVC(probability=True).
        ]
    )
    # Stratified splitting keeps the same class proportions in every fold as in the whole dataset.
    # random_state ensures reproducibility, and shuffle=True randomizes the data before splitting to avoid any order bias.
    cv = StratifiedKFold(n_splits=outer_splits, shuffle=True, random_state=config.random_state)
    # GridSearchCV exhaustively tries all combinations of the specified hyperparameters (C and gamma for the SVM) and evaluates each combination using cross-validation on the training data. It selects the combination that yields the best average F1 score across the folds.
    search = GridSearchCV(
        pipeline,
        param_grid=config.param_grid,
        scoring=config.scoring,
        cv=cv,
        n_jobs=config.n_jobs,
        verbose=config.verbose,
    )
    search.fit(X_train, y_train)
    return search


print(f"Training feature matrix: {X_train.shape}")
print(f"Testing feature matrix:  {X_test.shape}")

model = train_model(X_train, y_train, training_config)


# Evaluate the model on the test set
y_pred = model.predict(X_test)

report = classification_report(
    y_test,
    y_pred,
    target_names=CLASS_NAMES,
    output_dict=True,
    zero_division=0,
)
matrix = confusion_matrix(y_test, y_pred).tolist()
joblib.dump(model.best_estimator_, app_config.output_model)

print(f"Best parameters: {model.best_params_}")
print(f"Best CV F1: {model.best_score_:.4f}")
print(classification_report(y_test, y_pred, target_names=CLASS_NAMES, zero_division=0))
print("Confusion matrix:")
print(np.asarray(matrix))

model_svm = model
test_paths_svm = test_paths #path of images used for testing
y_test_svm = y_test #array of labels
X_test_svm = X_test #feature matrix of test images
y_pred_svm = y_pred #array of predicted labels
%store model_svm
%store test_paths_svm
%store y_test_svm
%store X_test_svm
%store y_pred_svm
%store feature_config




In [ ]:
metrics = {
    "best_params": model.best_params_ if hasattr(model, "best_params_") else None,
    "best_cv_f1": float(model.best_score_) if hasattr(model, "best_score_") else None,
    "classification_report": report if "report" in globals() else None,
    "confusion_matrix": matrix if "matrix" in globals() else None,
    "train_images": len(train_paths) if "train_paths" in globals() else None,
    "test_images": len(test_paths),
    "feature_count": int(X_train.shape[1]) if "X_train" in globals() else int(X_test.shape[1]),
    "feature_extraction_config": asdict(feature_config),
    "training_config": asdict(training_config) if "training_config" in globals() else None,
}
app_config.metrics_json.write_text(json.dumps(metrics, indent=2))

print(f"Saved model to:   {app_config.output_model.resolve()}")
print(f"Saved metrics to: {app_config.metrics_json.resolve()}")

# Bias and Diagnostic Analysis
- **Lighting Robustness Tests** assess whether the model’s predictions remain stable under realistic illumination perturbations.
- We apply controlled changes such as darkening, brightening, additive offset, and contrast shifts before the same CLAHE normalization and feature extraction pipeline.
- This diagnostic analysis checks for bias toward particular brightness or contrast conditions that could unfairly affect defect detection in industrial images.
- Advantages: it reveals whether model decisions depend on lighting artifacts rather than intrinsic texture features, and it quantifies prediction drift under operating variability.
- Limitations: these tests do not cover all real-world imaging variations (e.g. shadows, reflections, sensor noise), but they do provide a useful first check for illumination sensitivity.
- Practical interpretation: stable accuracy and low flip-rate under moderate lighting perturbations supports the claim that the model is more robust to illumination bias than a model trained without this normalization pipeline.
- Summary of results: the model passed the robustness criterion with min accuracy = 0.904 and max flip rate = 0.052 for darkening, additive offsets, and moderate contrast shifts.
- Claim: Model is robust to non-saturating lighting variation after CLAHE normalization.
- Criterion: For darkening, additive offsets, and moderate contrast shifts: accuracy >= 0.90 and flip_rate <= 0.06.
- Note: severe brightening is reported as a stress test because clipping can destroy texture information.

# Algorithmic Fairness
- In this defect inspection scenario, false positives and false negatives have distinct operational consequences.
- A false positive means a good part is flagged as defective, causing unnecessary rework, inspection cost, and lower manufacturing throughput.
- A false negative means a defective part is passed as good, which is more serious because it can lead to downstream failures, safety hazards, and customer dissatisfaction.
- For real-world fairness, the model should prioritize reducing false negatives while keeping false positives low enough to avoid excessive waste and mistrust in automated quality control.
- Confusion matrices, class-specific recall, and robustness checks under varied lighting help reveal whether the system is systematically biased toward one type of error.


In [ ]:
# Lighting robustness evaluation
# This tests whether the trained model keeps its predictions stable when test images are
# darkened, brightened, offset, or contrast-shifted before the same preprocessing pipeline.

def perturb_lighting(image: np.ndarray, perturbation: str) -> np.ndarray:
    if perturbation == "baseline":
        return image
    if perturbation == "dark_25":
        return np.clip(image * 0.75, 0.0, 1.0)
    if perturbation == "dark_50":
        return np.clip(image * 0.50, 0.0, 1.0)
    if perturbation == "bright_25":
        return np.clip(image * 1.25, 0.0, 1.0)
    if perturbation == "bright_50":
        return np.clip(image * 1.50, 0.0, 1.0)
    if perturbation == "offset_plus_10":
        return np.clip(image + 0.10, 0.0, 1.0)
    if perturbation == "offset_minus_10":
        return np.clip(image - 0.10, 0.0, 1.0)
    if perturbation == "low_contrast":
        return np.clip((image - 0.5) * 0.75 + 0.5, 0.0, 1.0)
    if perturbation == "high_contrast":
        return np.clip((image - 0.5) * 1.25 + 0.5, 0.0, 1.0)
    raise ValueError(f"Unknown lighting perturbation: {perturbation}")


def build_perturbed_feature_matrix(
    image_paths: list[Path],
    feature_config: FeatureExtractionConfig,
    perturbation: str,
) -> np.ndarray:
    features = []
    for image_path in image_paths:
        raw_image = read_grayscale_image(image_path, feature_config.image_size, config=None)
        perturbed_image = perturb_lighting(raw_image, perturbation)
        normalized_image = normalize_illumination(perturbed_image, feature_config)
        features.append(extract_features_from_image(normalized_image, feature_config))
    return np.vstack(features)


def evaluate_lighting_robustness(
    image_paths: list[Path],
    y_true: np.ndarray,
    baseline_predictions: np.ndarray,
    fitted_model: Pipeline,
    feature_config: FeatureExtractionConfig,
) -> list[dict[str, object]]:
    perturbations = [
        "baseline",
        "dark_25",
        "dark_50",
        "offset_plus_10",
        "offset_minus_10",
        "low_contrast",
        "high_contrast",
        "bright_25",
        "bright_50",
    ]
    normal_lighting_tests = {
        "dark_25",
        "dark_50",
        "offset_plus_10",
        "offset_minus_10",
        "low_contrast",
        "high_contrast",
    }
    rows = []

    for perturbation in perturbations:
        X_variant = build_perturbed_feature_matrix(
            image_paths,
            feature_config,
            perturbation,
        )
        y_variant = fitted_model.predict(X_variant)
        matrix_variant = confusion_matrix(y_true, y_variant, labels=[0, 1])
        accuracy = float(np.mean(y_variant == y_true))
        flips = int(np.sum(y_variant != baseline_predictions))
        good_recall = float(matrix_variant[0, 0] / matrix_variant[0].sum())
        not_good_recall = float(matrix_variant[1, 1] / matrix_variant[1].sum())
        rows.append(
            {
                "perturbation": perturbation,
                "accuracy": accuracy,
                "prediction_flips_vs_baseline": flips,
                "flip_rate_vs_baseline": float(flips / len(y_true)),
                "good_recall": good_recall,
                "not_good_recall": not_good_recall,
                "confusion_matrix": matrix_variant.tolist(),
                "included_in_robustness_claim": perturbation in normal_lighting_tests,
            }
        )

    return rows


lighting_robustness = evaluate_lighting_robustness(
    test_paths,
    y_test,
    y_pred,
    model.best_estimator_,
    feature_config,
)

print("Lighting robustness check")
print("perturbation       acc   flips  flip_rate  good_recall  not_good_recall")
for row in lighting_robustness:
    print(
        f"{row['perturbation']:16s} "
        f"{row['accuracy']:.3f} "
        f"{row['prediction_flips_vs_baseline']:5d} "
        f"{row['flip_rate_vs_baseline']:.3f} "
        f"{row['good_recall']:.3f} "
        f"{row['not_good_recall']:.3f}"
    )

claim_rows = [row for row in lighting_robustness if row["included_in_robustness_claim"]]
min_claim_accuracy = min(row["accuracy"] for row in claim_rows)
max_claim_flip_rate = max(row["flip_rate_vs_baseline"] for row in claim_rows)
lighting_robustness_summary = {
    "claim": "Model is robust to non-saturating lighting variation after CLAHE normalization.",
    "criterion": "For darkening, additive offsets, and moderate contrast shifts: accuracy >= 0.90 and flip_rate <= 0.06.",
    "passed": bool(min_claim_accuracy >= 0.90 and max_claim_flip_rate <= 0.06),
    "min_claim_accuracy": float(min_claim_accuracy),
    "max_claim_flip_rate": float(max_claim_flip_rate),
    "note": "Severe brightening is reported as a stress test because clipping can destroy texture information.",
}

print("\nRobustness summary")
print(json.dumps(lighting_robustness_summary, indent=2))

metrics_path = app_config.metrics_json
metrics_with_robustness = json.loads(metrics_path.read_text())
metrics_with_robustness["lighting_robustness"] = {
    "summary": lighting_robustness_summary,
    "results": lighting_robustness,
}
metrics_path.write_text(json.dumps(metrics_with_robustness, indent=2))
print(f"Saved lighting robustness results to: {metrics_path.resolve()}")



# Deployment Considerations

TODO add diagram and formating
🏗️ Architecture (Selected Design)
Containerized FastAPI service deployed on AWS SageMaker (real-time endpoint)

A Dockerized FastAPI API serves the model, accepting raw images as input.
The service bundles:

Preprocessing (GLCM + DWT feature extraction)
Normalization (StandardScaler)
SVC (RBF) inference


Deployed as a SageMaker real-time endpoint with CPU-based auto-scaling.

✅ Justification

FastAPI → low-latency, high-performance API suitable for real-time inference
Docker → ensures reproducibility and portability
SageMaker → managed deployment, scaling, and integration with monitoring
Bundled pipeline → prevents training–serving inconsistencies
CPU-only → cost-efficient for lightweight SVM inference


🧰 Tools & Libraries

Modeling & preprocessing: scikit-learn, joblib
API serving: FastAPI
Containerization: Docker
Model tracking/versioning: MLflow
Monitoring: Prometheus + Grafana (or SageMaker built-in tools)

✅ Justification

scikit-learn + joblib → simple, reliable pipeline serialization
FastAPI → modern, faster alternative to Flask
MLflow → experiment tracking and model registry support
Prometheus/Grafana → flexible, production-grade observability


📊 Observability & Monitoring

Track latency and throughput for system performance
Monitor prediction distribution and class-specific errors (FP/FN)
Implement drift detection:

Illumination drift (e.g., lighting condition changes)
Texture drift (e.g., manufacturing variations affecting GLCM/DWT features)


Log requests and predictions for debugging and quality control

✅ Justification

Ensures early detection of performance degradation
Identifies real-world data shifts affecting model reliability
Supports continuous quality assurance in industrial settings


🔐 Security

Validate and sanitize inputs (image format, resolution, value ranges)
Secure endpoints using API keys/OAuth + HTTPS/TLS
Apply rate limiting to prevent abuse and ensure stability


🔄 Lifecycle & Retraining

Collect labeled production data, especially failure cases
Retrain periodically or when performance metrics degrade
Use MLflow model versioning to manage updates
Deploy new models using A/B testing or shadow deployment before full rollout

✅ Justification

Keeps the model aligned with evolving production conditions
Reduces risk when deploying updates
Enables reproducibility and controlled experimentation
